In [ ]:
# Step 1: Import Libraries

import cv2  # مكتبة OpenCV لمعالجة الصور واكتشاف الأشكال
import numpy as np  # مكتبة للتعامل مع المصفوفات وإنشاء Kernel
from google.colab import files  # لرفع الصور من الجهاز
from google.colab.patches import cv2_imshow  # لعرض الصور في Google Colab

In [ ]:
# Step 2: Upload Image

uploaded = files.upload()  # فتح نافذة لاختيار الصورة ورفعها

image_path = next(iter(uploaded))  # الحصول على اسم الملف الذي تم رفعه

In [ ]:
# Step 3: Read and Display Image

image = cv2.imread(image_path)  # قراءة الصورة باستخدام OpenCV

# التأكد من أن الصورة تم تحميلها بنجاح
if image is None:
    raise ValueError("Could not load the image. Please upload a valid image.")

print("Image loaded:", image is not None)  # طباعة حالة تحميل الصورة
print("Image dimensions:", image.shape)  # طباعة الارتفاع والعرض وعدد القنوات

cv2_imshow(image)  # عرض الصورة الأصلية

In [ ]:
# Step 4: Convert Image to Grayscale

gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
# تحويل الصورة من الألوان إلى درجات الرمادي لتسهيل المعالجة

In [ ]:
# Step 5: Apply Gaussian Blur

blurred = cv2.GaussianBlur(gray, (5, 5), 0)
# تنعيم الصورة لتقليل تأثير التفاصيل الصغيرة والتشويش

In [ ]:
# Step 6: Detect Edges Using Canny

edges = cv2.Canny(blurred, 30, 100)
# اكتشاف الحواف التي قد تمثل حدود المستطيلات

print("Edges detected successfully")  # التأكد من تنفيذ الخطوة
cv2_imshow(edges)  # عرض الحواف المكتشفة

In [ ]:
# Step 7: Find Contours

contours, _ = cv2.findContours(
    edges,  # الصورة التي تحتوي على الحواف
    cv2.RETR_LIST,  # استخراج جميع الحدود دون بناء علاقات بينها
    cv2.CHAIN_APPROX_SIMPLE  # تقليل عدد النقاط المخزنة لكل حد
)

print("Number of contours detected:", len(contours))
# طباعة عدد الحدود التي تم اكتشافها

In [ ]:
# Step 8: Apply Otsu Thresholding

_, thresh = cv2.threshold(
    gray,  # صورة درجات الرمادي
    0,  # قيمة العتبة يتم تحديدها تلقائيًا باستخدام Otsu
    255,  # القيمة القصوى للبكسل
    cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
    # عكس الألوان وتحديد العتبة تلقائيًا
)

cv2_imshow(thresh)  # عرض الصورة الثنائية الناتجة

In [ ]:
# Step 9: Close Small Gaps

kernel = np.ones((3, 3), np.uint8)
# إنشاء Kernel بحجم 3×3 لاستخدامه في المعالجة المورفولوجية

closed = cv2.morphologyEx(
    thresh,  # الصورة الثنائية
    cv2.MORPH_CLOSE,  # إغلاق الفجوات الصغيرة داخل الخطوط
    kernel,  # العنصر المستخدم في المعالجة
    iterations=2  # تكرار العملية مرتين
)

cv2_imshow(closed)  # عرض الصورة بعد إغلاق الفجوات

In [ ]:
# Step 10: Find Processed Contours

contours2, _ = cv2.findContours(
    closed,  # الصورة بعد إغلاق الفجوات
    cv2.RETR_LIST,  # استخراج جميع الحدود
    cv2.CHAIN_APPROX_SIMPLE  # تبسيط تخزين نقاط الحدود
)

In [ ]:
# Step 11: Detect Rectangular Shapes

new_rectangles = []  # قائمة لتخزين المستطيلات المكتشفة

for contour in contours2:  # المرور على كل حد مكتشف

    perimeter = cv2.arcLength(contour, True)
    # حساب محيط الحد، مع اعتبار أن الشكل مغلق

    approx = cv2.approxPolyDP(
        contour,  # الحد المراد تبسيطه
        0.02 * perimeter,  # مقدار السماح بالاختلاف أثناء التبسيط
        True  # الحد مغلق
    )

    # التأكد من أن الشكل رباعي ومحدب
    if len(approx) == 4 and cv2.isContourConvex(approx):

        x, y, w, h = cv2.boundingRect(approx)
        # حساب موقع المستطيل وعرضه وارتفاعه

        # استبعاد الأشكال الصغيرة جدًا
        if w >= 30 and h >= 20:
            new_rectangles.append((x, y, w, h))
            # إضافة إحداثيات المستطيل إلى القائمة

print("Rectangles before removing duplicates:", len(new_rectangles))
# طباعة عدد المستطيلات قبل إزالة التكرارات

In [ ]:
# Step 12: Calculate Intersection over Union (IoU)

def calculate_iou(box1, box2):

    # استخراج إحداثيات وأبعاد المستطيل الأول
    x1, y1, w1, h1 = box1

    # استخراج إحداثيات وأبعاد المستطيل الثاني
    x2, y2, w2, h2 = box2

    # تحديد حدود منطقة التقاطع بين المستطيلين
    x_left = max(x1, x2)
    y_top = max(y1, y2)
    x_right = min(x1 + w1, x2 + w2)
    y_bottom = min(y1 + h1, y2 + h2)

    # حساب عرض وارتفاع منطقة التقاطع
    intersection_width = max(0, x_right - x_left)
    intersection_height = max(0, y_bottom - y_top)

    # حساب مساحة التقاطع
    intersection_area = intersection_width * intersection_height

    # حساب مساحة كل مستطيل
    area1 = w1 * h1
    area2 = w2 * h2

    # حساب مساحة الاتحاد بين المستطيلين
    union_area = area1 + area2 - intersection_area

    # تجنب القسمة على صفر
    if union_area == 0:
        return 0

    # إرجاع نسبة التقاطع إلى الاتحاد
    return intersection_area / union_area

In [ ]:
# Step 13: Remove Duplicate Rectangles

unique_rectangles2 = []  # قائمة المستطيلات النهائية دون التكرارات

# ترتيب المستطيلات من الأكبر مساحة إلى الأصغر
for box in sorted(
    new_rectangles,
    key=lambda b: b[2] * b[3],
    reverse=True
):

    duplicate = False  # افتراض أن المستطيل غير مكرر

    # مقارنة المستطيل الحالي بالمستطيلات التي احتفظنا بها
    for kept_box in unique_rectangles2:

        overlap = calculate_iou(box, kept_box)
        # حساب نسبة التداخل بين المستطيلين

        # إذا كان التداخل أكبر من 80% نعتبره مكررًا
        if overlap > 0.8:
            duplicate = True
            break  # إيقاف المقارنة لهذا المستطيل

    # الاحتفاظ بالمستطيل إذا لم يكن مكررًا
    if not duplicate:
        unique_rectangles2.append(box)

print("Rectangles after removing duplicates:", len(unique_rectangles2))
# طباعة عدد المستطيلات بعد إزالة التكرارات

In [ ]:
# Step 14: Print Rectangle Coordinates

print("\nDetected Shapes and Coordinates:")
# طباعة عنوان النتائج

for i, (x, y, w, h) in enumerate(unique_rectangles2, start=1):
    # المرور على المستطيلات وترقيمها بداية من 1

    print(f"Shape {i}:")  # طباعة رقم الشكل
    print(f"  X = {x}")  # الإحداثي الأفقي للزاوية العلوية اليسرى
    print(f"  Y = {y}")  # الإحداثي الرأسي للزاوية العلوية اليسرى
    print(f"  Width = {w}")  # عرض المستطيل
    print(f"  Height = {h}")  # ارتفاع المستطيل
    print("-" * 25)  # طباعة خط فاصل بين النتائج

In [ ]:
# Step 15: Draw Rectangles and Labels

final_image = image.copy()
# إنشاء نسخة من الصورة الأصلية للحفاظ عليها دون تغيير

for i, (x, y, w, h) in enumerate(unique_rectangles2, start=1):
    # المرور على المستطيلات النهائية

    cv2.rectangle(
        final_image,  # الصورة التي سنرسم عليها
        (x, y),  # إحداثيات الزاوية العلوية اليسرى
        (x + w, y + h),  # إحداثيات الزاوية السفلية اليمنى
        (0, 255, 0),  # اللون الأخضر بصيغة BGR
        3  # سُمك الخط
    )

    # كتابة رقم الشكل فوق المستطيل
    cv2.putText(
        final_image,  # الصورة التي ستظهر عليها الكتابة
        f"Shape {i}",  # النص الذي يحتوي على رقم الشكل
        (x, max(y - 10, 20)),  # مكان كتابة النص
        cv2.FONT_HERSHEY_SIMPLEX,  # نوع الخط
        0.7,  # حجم الخط
        (0, 0, 255),  # اللون الأحمر بصيغة BGR
        2  # سُمك الخط
    )

In [ ]:
# Step 16: Display Final Result

print("\nFinal number of rectangles:", len(unique_rectangles2))
# طباعة العدد النهائي للمستطيلات المكتشفة

cv2_imshow(final_image)  # عرض الصورة النهائية

In [ ]:
# Step 17: Save Final Image

output_path = "/content/final_detected_rectangles.jpg"
# تحديد مكان حفظ الصورة واسم الملف

cv2.imwrite(output_path, final_image)
# حفظ الصورة النهائية على هيئة ملف JPG

print("Final image saved:", output_path)
# طباعة مسار الصورة المحفوظة